In [2]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import math

# для заданий 1 и 2 (n = 24)
data_24 = np.array([
    -18.55, -18.02, -17.65, -15.84, -15.50, -14.09, -13.07, -12.20,
    -11.78, -11.39, -9.61, -6.45, -0.93, -0.72, -0.51, 0.56,
    0.83, 1.55, 1.67, 4.64, 8.72, 9.90, 14.29, 17.22
])

# для заданий 3 и 4 (n = 31)
data_31 = np.array([
    -21.08, -18.55, -18.02, -17.65, -15.84, -15.50, -14.09, -13.07,
    -12.20, -11.78, -11.39, -9.61, -8.95, -6.45, -3.13, -2.30,
    -0.93, -0.72, -0.51, 0.56, 0.83, 1.55, 1.67, 2.68,
    3.89, 4.64, 5.13, 8.72, 9.90, 14.29, 17.22
])

print("\n--- задание 1 И 2 (n=24) ---")

n24 = len(data_24)
mean24 = np.mean(data_24)
# с ddof=1 работает как в Excel ДИСП.В
var24 = np.var(data_24, ddof=1)
std24 = np.std(data_24, ddof=1)
median24 = np.median(data_24)

# асимметрия и эксцесс (bias=False делает их "выборочными", как в Excel)
skew24 = stats.skew(data_24, bias=False)
kurt24 = stats.kurtosis(data_24, bias=False)

# усеченное среднее 
trim_mean24 = stats.trim_mean(data_24, 0.05)

# коэффициент вариации и относ. линейное отклонение
cv24 = (std24 / abs(mean24)) * 100
mad24 = np.mean(np.abs(data_24 - mean24)) # среднее абсолютное отклонение
rld24 = (mad24 / abs(mean24)) * 100

print(f"Среднее значение: {mean24:.4f}")
print(f"Дисперсия (выборочная): {var24:.4f}")
print(f"Ср. кв. отклонение (S): {std24:.4f}")
print(f"Медиана: {median24:.4f}")
print("Мода: Отсутствует (все значения уникальны)")
print(f"Коэфф. асимметрии: {skew24:.4f}")
print(f"Коэфф. эксцесса: {kurt24:.4f}")
print(f"Усеченное среднее (10%): {trim_mean24:.4f}")
print(f"Коэффициент вариации: {cv24:.2f} %")
print(f"Относ. линейное отклонение: {rld24:.2f} %")


print("\n--- задание 3: доверительный интервал (n=31) ---")

n31 = len(data_31)
mean31 = np.mean(data_31)
std31 = np.std(data_31, ddof=1)
alpha = 0.05

# предельная ошибка
margin_of_error = stats.norm.ppf(1 - alpha/2) * (std31 / np.sqrt(n31))

lower_bound = mean31 - margin_of_error
upper_bound = mean31 + margin_of_error

print(f"среднее: {mean31:.4f}")
print(f"ср. кв. отклонение: {std31:.4f}")
print(f"предельная ошибка: {margin_of_error:.4f}")
print(f"доверительный интервал: ({lower_bound:.4f} ; {upper_bound:.4f})")


print("\n--- задание 4: критерий пирсена ---")

# параметры интервалов
min_val = np.min(data_31)
max_val = np.max(data_31)
R = max_val - min_val
k = round(1 + 3.322 * math.log10(n31))
h = R / k

print(f"размах (R): {R:.4f}")
print(f"количество интервалов (k): {k}")
print(f"шаг: {h:.4f}\n")

# создаем границы интервалов и считаем частоты
bins = [min_val + i*h for i in range(k+1)]

# np.histogram сама считает, сколько попало в интервалы
obs_freq, _ = np.histogram(data_31, bins=bins)

# считаем теоретические частоты и Хи-квадрат
chi_square_obs = 0
print(f"{'№':<3} | {'середина (xi)':<13} | {'эмпирическая частота. (mi)':<10} | {'теоретическая (mi_teor)':<13} | {'Хи-квадрат':<10}")
print("-" * 60)

for i in range(k):
    # середина интервала
    midpoint = (bins[i] + bins[i+1]) / 2
    
    # теоретическая частота
    pdf_val = stats.norm.pdf(midpoint, mean31, std31)
    teor_freq = h * n31 * pdf_val
    
    # Хи-квадрат
    chi_sq_component = ((obs_freq[i] - teor_freq)**2) / teor_freq
    chi_square_obs += chi_sq_component
    
    print(f"{i+1:<3} | {midpoint:<13.3f} | {obs_freq[i]:<10} | {teor_freq:<13.3f} | {chi_sq_component:<10.3f}")

print("-" * 60)

# проверка гипотезы
df = k - 3 # степени свободы
chi_square_crit = stats.chi2.ppf(1 - alpha, df) # критическое значение

print(f"наблюдаемый Хи-квадрат: {chi_square_obs:.3f}")
print(f"степени свободы (df): {df}")
print(f"критический Хи-квадрат: {chi_square_crit:.3f}")

if chi_square_obs < chi_square_crit:
    print("\nнаблюдаемое значение МЕНЬШЕ критического.")
    print("гипотеза о нормальном распределении ПРИНИМАЕТСЯ.")
else:
    print("\nнаблюдаемое значение БОЛЬШЕ критического.")
    print("гипотеза о нормальном распределении ОТВЕРГАЕТСЯ.")


--- задание 1 И 2 (n=24) ---
Среднее значение: -4.4554
Дисперсия (выборочная): 113.9078
Ср. кв. отклонение (S): 10.6728
Медиана: -3.6900
Мода: Отсутствует (все значения уникальны)
Коэфф. асимметрии: 0.4042
Коэфф. эксцесса: -0.8792
Усеченное среднее (10%): -4.8000
Коэффициент вариации: 239.55 %
Относ. линейное отклонение: 207.02 %

--- задание 3: доверительный интервал (n=31) ---
среднее: -4.2158
ср. кв. отклонение: 10.2201
предельная ошибка: 3.5977
доверительный интервал: (-7.8135 ; -0.6181)

--- задание 4: критерий пирсена ---
размах (R): 38.3000
количество интервалов (k): 6
шаг: 6.3833

№   | середина (xi) | эмпирическая частота. (mi) | теоретическая (mi_teor) | Хи-квадрат
------------------------------------------------------------
1   | -17.888       | 6          | 3.157         | 2.561     
2   | -11.505       | 7          | 5.990         | 0.170     
3   | -5.122        | 3          | 7.694         | 2.864     
4   | 1.262         | 9          | 6.691         | 0.797     
5   | 